# 🤖 Notebook 3 — ML Modeling & Explainability
**Auto Insurance Churn Project**

### Goals
1. Train a **Logistic Regression** baseline (simple, interpretable)
2. Train an **XGBoost** model (high performance)
3. Handle class imbalance with **SMOTE**
4. Evaluate with **ROC-AUC**, **Precision-Recall**, and **F1**
5. Explain predictions with **SHAP values**

### Why these metrics?
Accuracy is misleading on imbalanced data. If 12% churn:
- A model that predicts "never churn" gets 88% accuracy but catches 0 churners
- **ROC-AUC** measures how well the model ranks churners above non-churners
- **Recall (churn class)** tells us: of actual churners, how many did we catch?
- **Precision (churn class)** tells us: of predicted churners, how many were right?


In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import shap

from features import get_model_features
from evaluate import print_report, plot_confusion_matrix, plot_roc_curve, plot_precision_recall

# Load model-ready data (output of notebook 02)
PROCESSED = Path("../data/processed/churn_model_ready.csv")
df = pd.read_csv(PROCESSED)
print(f"Loaded: {df.shape}")
print(f"Churn rate: {df['churn'].mean()*100:.2f}%")


## 1. Train / Test Split

We split 80/20 with stratification to preserve the churn ratio in both sets.

In [ ]:
feature_cols = get_model_features()
X = df[feature_cols]
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train churn rate: {y_train.mean()*100:.2f}%")
print(f"Test churn rate:  {y_test.mean()*100:.2f}%")


## 2. Handle Class Imbalance with SMOTE

**SMOTE** (Synthetic Minority Over-sampling Technique) generates synthetic
examples of the minority class (churners) so the model sees a more balanced
training set. We apply it ONLY to training data — never to the test set.

Why? Because the test set must reflect real-world distribution.


In [ ]:
smote = SMOTE(random_state=42, sampling_strategy=0.4)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print(f"After SMOTE — Train size: {X_train_sm.shape}")
print(f"Churn rate after SMOTE: {y_train_sm.mean()*100:.2f}%")


## 3. Baseline — Logistic Regression

Always start with the simplest model. It sets a performance floor and is highly interpretable.

In [ ]:
lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),  # LR needs scaled features
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))
])

lr_pipeline.fit(X_train_sm, y_train_sm)

y_pred_lr = lr_pipeline.predict(X_test)
y_prob_lr = lr_pipeline.predict_proba(X_test)[:, 1]

print_report(y_test, y_pred_lr, y_prob_lr, "Logistic Regression")
plot_roc_curve(y_test, y_prob_lr, "Logistic Regression")


## 4. XGBoost Classifier

XGBoost is a gradient-boosted tree model — typically the top performer on tabular data. It handles non-linear relationships and feature interactions automatically.

In [ ]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(y_train.value_counts()[0] / y_train.value_counts()[1]),
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=50)

y_pred_xgb = xgb.predict(X_test)
y_prob_xgb = xgb.predict_proba(X_test)[:, 1]

print_report(y_test, y_pred_xgb, y_prob_xgb, "XGBoost")
plot_roc_curve(y_test, y_prob_xgb, "XGBoost")
plot_precision_recall(y_test, y_prob_xgb, "XGBoost")


## 5. SHAP — Model Explainability

SHAP (SHapley Additive exPlanations) tells us **why** the model made each prediction.

- **Global SHAP:** Which features matter most across all customers?
- **Individual SHAP:** Why did the model flag THIS customer as high-risk?

This is critical for business adoption — a model no one can explain won't be trusted.


In [ ]:
# Compute SHAP values on a sample (full dataset can be slow)
sample_idx = X_test.sample(2000, random_state=42).index
X_sample = X_test.loc[sample_idx]

explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_sample)

# Global feature importance
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_sample, feature_names=feature_cols,
                  plot_type="bar", show=False)
plt.title("SHAP Feature Importance — XGBoost Churn Model", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/figures/03_shap_importance.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Beeswarm plot — shows direction and magnitude of each feature
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_sample, feature_names=feature_cols, show=False)
plt.title("SHAP Beeswarm — Feature Impact on Churn Probability", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/figures/03_shap_beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. Model Comparison

In [ ]:
from sklearn.metrics import f1_score, average_precision_score

results = pd.DataFrame({
    "Model": ["Logistic Regression", "XGBoost"],
    "ROC-AUC": [
        roc_auc_score(y_test, y_prob_lr).round(4),
        roc_auc_score(y_test, y_prob_xgb).round(4)
    ],
    "PR-AUC": [
        average_precision_score(y_test, y_prob_lr).round(4),
        average_precision_score(y_test, y_prob_xgb).round(4)
    ],
    "F1 (Churn)": [
        f1_score(y_test, y_pred_lr, pos_label=1).round(4),
        f1_score(y_test, y_pred_xgb, pos_label=1).round(4)
    ]
})
print(results.to_string(index=False))


## Summary

- XGBoost outperforms Logistic Regression on all metrics for this imbalanced dataset
- SHAP confirms that **tenure**, **annual premium**, and **good credit** are the top 3 churn drivers
- The model can be used to score all active customers and prioritize retention outreach

### Business Application
Run the model monthly on the full customer base. Flag customers with predicted churn probability > 0.4 for proactive outreach — a discount offer, policy review call, or loyalty reward.
